In [1]:
!mkdir /kaggle/temp

In [2]:
!wget -O /kaggle/temp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

--2026-06-18 18:52:21--  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-Linux-x86_64.sh [following]
--2026-06-18 18:52:22--  https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-Linux-x86_64.sh
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/221584272/dd517a4f-612c-449e-89ad-3755d98f8038?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-18T19%3A30%3A28Z&rscd=attachment%3B+filename%3DMiniforge3-Linux-x86_64.sh&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=202

In [3]:
!rm -rf /kaggle/temp/conda
!bash /kaggle/temp/Miniforge3.sh -b -p /kaggle/temp/conda

PREFIX=/kaggle/temp/conda
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.5.20-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_19.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_102.conda
Extracting libgcc-15.2.0-he0feb66_19.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.8.1-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_19.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libsqlite-3.53.1-h0c1763c_0.c

In [4]:
!/kaggle/temp/conda/bin/mamba install -y --quiet pytorch=2.11 pillow matplotlib scipy matplotlib-inline ipython nvimgcodec cuda-toolkit nccl

warning  libmamba [gdb-16.3-py313h80c3f9b_6] The following files were already present in the environment:
    - share/info/bfd.info
    - share/info/ctf-spec.info
    - share/info/sframe-spec.info
warning  libmamba [cuda-gdb-13.3.27-hba53cbc_0] The following files were already present in the environment:
    - targets/x86_64-linux/include/cuda_stdint.h


In [5]:
!rm -rf /kaggle/temp/music_deep
!git clone https://github.com/kwon-young/music_deep.git /kaggle/temp/music_deep

Cloning into '/kaggle/temp/music_deep'...
remote: Enumerating objects: 2095, done.
remote: Counting objects: 100% (2095/2095), done.
remote: Compressing objects: 100% (790/790), done.
remote: Total 2095 (delta 1424), reused 1920 (delta 1265), pack-reused 0 (from 0)
Receiving objects: 100% (2095/2095), 673.55 KiB | 7.65 MiB/s, done.
Resolving deltas: 100% (1424/1424), done.


In [6]:
!PYTHONPATH=/kaggle/temp/music_deep /kaggle/temp/conda/bin/mamba run torchrun --nproc_per_node=2 /kaggle/temp/music_deep/src/train_detection.py \
    --exp_dir experiments/016_vit_small_baseline \
    --patch_size 64 \
    --epochs 10 \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --headless \
    --cache_dir /kaggle/temp/cache/ \
    --use_sdpa \
    --compile \
    --log_epoch_interval 0.5 \
    --backbone_size small

W0618 18:54:04.277000 987 site-packages/torch/distributed/run.py:851] 
W0618 18:54:04.277000 987 site-packages/torch/distributed/run.py:851] *****************************************
W0618 18:54:04.277000 987 site-packages/torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0618 18:54:04.277000 987 site-packages/torch/distributed/run.py:851] *****************************************
Parsing COCO JSON from ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json (this might take a while)...
Parsing COCO JSON from ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json (this might take a while)...
Caching parsed dataset to /kaggle/temp/cache/instances_trainval2017.pkl
Using prep_device: cuda:0
Using train_device: cuda:0
Using match_device: cuda:0
L

In [7]:
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/scripts/inference_coco.py \
    --checkpoint experiments/016_vit_small_baseline/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/016_vit_small_baseline/inference \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa \
    --backbone_size small

Using device: cuda
Loading cached COCO dataset from /kaggle/temp/cache/instances_trainval2017.pkl
Dynamic Top-K limits -> Symbols: 1268, Lines: 1785
Loading checkpoint from experiments/016_vit_small_baseline/train_detection/checkpoints/latest_model.pt
Running inference...
100%|█████████████████████████████████████████| 351/351 [04:58<00:00,  1.18it/s]
Saving 445068 symbol predictions to experiments/016_vit_small_baseline/inference/preds_symbols.json
Saving 626180 line predictions to experiments/016_vit_small_baseline/inference/preds_lines.json
